# Pdx1 target-edge dynamics along pancreatic endocrine pseudotime

This notebook reconstructs the requested endocrine trajectory:

**Ngn3 low EP → Ngn3 high EP → Pre-endocrine → Beta**

It aligns `pseudotime.csv`, `cell_data.csv`, and `cell{i}.npz` by their original row/cell index, sorts the selected cells by pseudotime, retains the **50 highest-weight edges globally in each single-cell GRN**, and visualizes the top-50 regulatory strength of Pdx1 toward eight specified targets.

Figure contract:

- **Core conclusion:** Pdx1–target regulatory strengths show target-specific dynamics along endocrine differentiation.
- **Evidence:** all 2,087 cells in the four requested states are used.
- **Top-50 rule:** the 50 largest positive edge weights across the complete TF × target matrix of each cell are retained.
- **Absent edge rule:** a requested Pdx1 edge not present among a cell's top 50 edges is assigned strength 0.
- **Curve:** Gaussian-kernel local mean; the shaded region is a descriptive local 95% standard-error band.
- **Stage guides:** faint backgrounds are divided at midpoints between adjacent stage median pseudotimes; overlapping cell states are retained and not reassigned.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the scDGRN repository.")


REPO_ROOT = find_repository_root()
sys.path.insert(0, str(REPO_ROOT))

DATASET_NAME = "pancreas"
RESULT_NAME = os.environ.get("SCDGRN_RESULT_NAME", "pancreas")
DATASETS_ROOT = Path(os.environ.get("SCDGRN_DATASETS_ROOT", REPO_ROOT / "datasets")).resolve()
RESULTS_ROOT = Path(os.environ.get("SCDGRN_RESULTS_ROOT", REPO_ROOT / "results")).resolve()
DATASET_DIR = DATASETS_ROOT / DATASET_NAME
RESULT_DIR = RESULTS_ROOT / RESULT_NAME
OUTPUT_DIR = RESULT_DIR / "tutorial_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMN = "celltype"
RANDOM_SEED = 3407
TOP_N_EDGES = 50

print(f"Repository: {REPO_ROOT}")
print(f"Dataset:    {DATASET_DIR}")
print(f"Results:    {RESULT_DIR}")
print(f"Outputs:    {OUTPUT_DIR}")


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy import sparse

import matplotlib
matplotlib.use("Agg")
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings("ignore", category=FutureWarning)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 7.5,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 0.75,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "legend.frameon": False,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "savefig.facecolor": "white",
})

print("Python plotting environment loaded.")


In [ ]:
# Paths and analysis parameters
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = REPO_ROOT
DATASET_DIR = PROJECT_ROOT / "datasets" / "pancreas"
NETWORK_DIR = RESULT_DIR / "single_network"
OUTPUT_DIR = RESULT_DIR / "tutorial_outputs" / "pdx1_pseudotime"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE_ORDER = ["Ngn3 low EP", "Ngn3 high EP", "Pre-endocrine", "Beta"]
TARGETS = ["Tcf12", "Zmiz1", "Rfx6", "Asb4", "Creb1", "Gadd45a", "Pax4", "Cpa2"]
REGULATOR = "Pdx1"
TOP_N = 50
SMOOTH_BANDWIDTH = 0.035
REBUILD_TOP50 = True

STAGE_COLORS = {
    "Ngn3 low EP": "#3A9B45",
    "Ngn3 high EP": "#005B78",
    "Pre-endocrine": "#B13216",
    "Beta": "#5B568F",
}

# Pale stage fills are used identically in the axes and legend.
STAGE_FILL_COLORS = {
    "Ngn3 low EP": "#E9F4EB",
    "Ngn3 high EP": "#E3EDF0",
    "Pre-endocrine": "#F6E8E5",
    "Beta": "#EDECF3",
}
CURVE_COLOR = "#172A46"
CI_COLOR = "#8CA5C2"

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")


## 1. Align cells and order the requested trajectory

The three cell-indexed inputs are required to have identical row counts. Cell identities are preserved; only the four requested endocrine states are selected and then stably sorted by pseudotime.


In [ ]:
cell_types = pd.read_csv(DATASET_DIR / "cell_data.csv")
pseudotime = pd.read_csv(DATASET_DIR / "pseudotime.csv")
expression_index = pd.read_csv(DATASET_DIR / "ExpressionData.csv", index_col=0, usecols=range(1)).index.astype(str)

if "celltype" not in cell_types.columns:
    raise KeyError("cell_data.csv must contain a 'celltype' column.")
if "pseudotime" not in pseudotime.columns:
    raise KeyError("pseudotime.csv must contain a 'pseudotime' column.")
if not (len(cell_types) == len(pseudotime) == len(expression_index)):
    raise ValueError(
        f"Cell count mismatch: cell_data={len(cell_types)}, pseudotime={len(pseudotime)}, "
        f"ExpressionData={len(expression_index)}"
    )

metadata = pd.DataFrame({
    "cell_index": np.arange(len(cell_types), dtype=int),
    "barcode": expression_index.to_numpy(),
    "celltype": cell_types["celltype"].astype(str).to_numpy(),
    "pseudotime": pd.to_numeric(pseudotime["pseudotime"], errors="raise").to_numpy(),
})

selected = metadata.loc[metadata["celltype"].isin(STAGE_ORDER)].copy()
selected["celltype"] = pd.Categorical(selected["celltype"], categories=STAGE_ORDER, ordered=True)
selected = selected.sort_values(["pseudotime", "cell_index"], kind="mergesort").reset_index(drop=True)
selected.insert(0, "trajectory_order", np.arange(len(selected), dtype=int))

if selected["pseudotime"].isna().any():
    raise ValueError("Selected pseudotime values contain missing data.")
if not selected["pseudotime"].is_monotonic_increasing:
    raise AssertionError("Selected cells are not monotonically ordered by pseudotime.")

stage_stats = (
    selected.groupby("celltype", observed=True)["pseudotime"]
    .agg(["count", "min", "mean", "median", "max"])
    .reindex(STAGE_ORDER)
)
if not stage_stats["median"].is_monotonic_increasing:
    raise ValueError("Stage median pseudotimes do not follow the requested biological order.")

selected_path = OUTPUT_DIR / "selected_cells_sorted_by_pseudotime.csv"
selected.to_csv(selected_path, index=False)
print(stage_stats.to_string(float_format=lambda x: f"{x:.4f}"))
print(f"\nSelected {len(selected):,} of {len(metadata):,} cells; saved: {selected_path}")


## 2. Retain the global top 50 edges in every selected single-cell GRN

The GRN matrix rows follow `TF.csv`, columns follow the gene columns of `ExpressionData.csv`, and `cell{i}.npz` follows the original cell row index. Ranking is descending by the stored non-negative regulatory strength; no absolute-value transformation or per-regulator ranking is applied.


In [ ]:
gene_names = pd.read_csv(DATASET_DIR / "ExpressionData.csv", index_col=0, nrows=1).columns.astype(str).to_numpy()
tf_names = pd.read_csv(DATASET_DIR / "TF.csv", header=None).iloc[:, 0].astype(str).to_numpy()

if REGULATOR not in set(tf_names):
    raise KeyError(f"Regulator {REGULATOR!r} was not found in TF.csv.")
missing_targets = [target for target in TARGETS if target not in set(gene_names)]
if missing_targets:
    raise KeyError(f"Targets missing from ExpressionData.csv: {missing_targets}")

network_files = list(NETWORK_DIR.glob("cell*.npz"))
if len(network_files) != len(metadata):
    raise ValueError(f"Expected {len(metadata)} single-cell networks, found {len(network_files)}.")

top50_path = OUTPUT_DIR / "selected_cells_top50_edges.csv.gz"
if REBUILD_TOP50 or not top50_path.exists():
    expected_shape = (len(tf_names), len(gene_names))
    capacity = len(selected) * TOP_N
    cell_idx_out = np.empty(capacity, dtype=np.int32)
    rank_out = np.empty(capacity, dtype=np.int16)
    strength_out = np.empty(capacity, dtype=np.float32)
    source_out = np.empty(capacity, dtype=object)
    target_out = np.empty(capacity, dtype=object)
    cursor = 0

    for position, row in enumerate(selected.itertuples(index=False), start=1):
        matrix_path = NETWORK_DIR / f"cell{int(row.cell_index)}.npz"
        matrix = sparse.load_npz(matrix_path).tocoo(copy=False)
        if matrix.shape != expected_shape:
            raise ValueError(f"{matrix_path.name}: expected {expected_shape}, found {matrix.shape}")
        finite = np.isfinite(matrix.data)
        if not finite.all():
            matrix = sparse.coo_matrix(
                (matrix.data[finite], (matrix.row[finite], matrix.col[finite])), shape=matrix.shape
            )
        if matrix.nnz < TOP_N:
            raise ValueError(f"{matrix_path.name} contains only {matrix.nnz} non-zero edges.")

        chosen = np.argpartition(matrix.data, -TOP_N)[-TOP_N:]
        chosen = chosen[np.argsort(matrix.data[chosen], kind="mergesort")[::-1]]
        sl = slice(cursor, cursor + TOP_N)
        cell_idx_out[sl] = int(row.cell_index)
        rank_out[sl] = np.arange(1, TOP_N + 1, dtype=np.int16)
        strength_out[sl] = matrix.data[chosen].astype(np.float32)
        source_out[sl] = tf_names[matrix.row[chosen]]
        target_out[sl] = gene_names[matrix.col[chosen]]
        cursor += TOP_N

        if position % 250 == 0 or position == len(selected):
            print(f"Processed {position:,}/{len(selected):,} selected cells")

    top_edges = pd.DataFrame({
        "cell_index": cell_idx_out[:cursor],
        "rank": rank_out[:cursor],
        "source": source_out[:cursor],
        "target": target_out[:cursor],
        "strength": strength_out[:cursor],
    })
    top_edges = top_edges.merge(
        selected[["trajectory_order", "cell_index", "barcode", "celltype", "pseudotime"]],
        on="cell_index", how="left", validate="many_to_one"
    )
    top_edges = top_edges[
        ["trajectory_order", "cell_index", "barcode", "celltype", "pseudotime", "rank", "source", "target", "strength"]
    ].sort_values(["trajectory_order", "rank"], kind="mergesort")
    top_edges.to_csv(top50_path, index=False, compression="gzip")
else:
    top_edges = pd.read_csv(top50_path)

edge_counts = top_edges.groupby("cell_index").size()
if len(edge_counts) != len(selected) or not edge_counts.eq(TOP_N).all():
    raise AssertionError("Every selected cell must contribute exactly 50 edges.")

print(f"Retained {len(top_edges):,} edges = {len(selected):,} cells × {TOP_N} edges.")
print(f"Saved: {top50_path}")


## 3. Build the Pdx1-target pseudotime table

Each selected cell contributes one observation for every requested target. If the Pdx1→target edge is not in that cell's global top 50, its top-50 regulatory strength is zero. This preserves all selected cells in the smoothed trajectory.


In [ ]:
cell_target_grid = selected.assign(_key=1).merge(
    pd.DataFrame({"target": TARGETS, "_key": 1}), on="_key", how="inner"
).drop(columns="_key")

pdx1_edges = top_edges.loc[
    top_edges["source"].eq(REGULATOR) & top_edges["target"].isin(TARGETS),
    ["cell_index", "target", "rank", "strength"]
].copy()

trajectory = cell_target_grid.merge(
    pdx1_edges, on=["cell_index", "target"], how="left", validate="one_to_one"
)
trajectory["is_top50"] = trajectory["rank"].notna()
trajectory["strength"] = trajectory["strength"].fillna(0.0).astype(float)
trajectory["rank"] = trajectory["rank"].astype("Int64")
trajectory["target"] = pd.Categorical(trajectory["target"], categories=TARGETS, ordered=True)
trajectory = trajectory.sort_values(["target", "pseudotime", "cell_index"], kind="mergesort")

trajectory_path = OUTPUT_DIR / "pdx1_target_top50_strength_by_pseudotime.csv"
trajectory.to_csv(trajectory_path, index=False)

summary = (
    trajectory.groupby(["target", "celltype"], observed=True)
    .agg(
        n_cells=("cell_index", "size"),
        top50_count=("is_top50", "sum"),
        top50_fraction=("is_top50", "mean"),
        mean_strength_all_cells=("strength", "mean"),
        median_strength_all_cells=("strength", "median"),
    )
    .reset_index()
)
present_means = (
    trajectory.loc[trajectory["is_top50"]]
    .groupby(["target", "celltype"], observed=True)["strength"]
    .mean().rename("mean_strength_when_present").reset_index()
)
summary = summary.merge(present_means, on=["target", "celltype"], how="left")
summary_path = OUTPUT_DIR / "pdx1_target_top50_stage_summary.csv"
summary.to_csv(summary_path, index=False)

overall = trajectory.groupby("target", observed=True).agg(
    n_cells=("cell_index", "size"),
    top50_count=("is_top50", "sum"),
    top50_fraction=("is_top50", "mean"),
    mean_strength=("strength", "mean"),
)
print(overall.to_string(float_format=lambda x: f"{x:.4f}"))
print(f"\nSaved: {trajectory_path}")
print(f"Saved: {summary_path}")


## 4. Plot Pdx1 regulatory-strength trajectories

Curves include every selected cell, including zeros for absent top-50 edges. Panel-specific y-axis ranges preserve target-specific temporal structure; all axes show the original, untransformed strength scale.


In [ ]:
def gaussian_local_mean_ci(x, y, grid, bandwidth):
    # Gaussian-kernel local mean and descriptive 95% local SE interval.
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    distance = (x[:, None] - grid[None, :]) / float(bandwidth)
    weights = np.exp(-0.5 * distance**2)
    weight_sum = weights.sum(axis=0)
    mean = (weights * y[:, None]).sum(axis=0) / weight_sum
    variance = (weights * (y[:, None] - mean[None, :])**2).sum(axis=0) / weight_sum
    effective_n = weight_sum**2 / np.square(weights).sum(axis=0)
    se = np.sqrt(np.maximum(variance, 0) / np.maximum(effective_n, 1))
    low = np.clip(mean - 1.96 * se, 0, None)
    high = mean + 1.96 * se
    return mean, low, high


x_min = float(selected["pseudotime"].min())
x_max = float(selected["pseudotime"].max())
grid = np.linspace(x_min, x_max, 260)
stage_medians = stage_stats["median"].to_numpy(dtype=float)
stage_boundaries = np.r_[x_min, (stage_medians[:-1] + stage_medians[1:]) / 2, x_max]

fig, axes = plt.subplots(4, 2, figsize=(7.25, 8.55), sharex=True, constrained_layout=False)
axes = axes.ravel()
panel_letters = list("abcdefgh")

for ax, target, panel_letter in zip(axes, TARGETS, panel_letters):
    data = trajectory.loc[trajectory["target"].astype(str).eq(target)].sort_values("pseudotime")
    x = data["pseudotime"].to_numpy(dtype=float)
    y = data["strength"].to_numpy(dtype=float)
    mean, low, high = gaussian_local_mean_ci(x, y, grid, SMOOTH_BANDWIDTH)

    for stage, left, right in zip(STAGE_ORDER, stage_boundaries[:-1], stage_boundaries[1:]):
        ax.axvspan(left, right, color=STAGE_FILL_COLORS[stage], alpha=1.0, lw=0, zorder=0)

    ax.fill_between(grid, low, high, color=CI_COLOR, alpha=0.30, lw=0, zorder=2)
    ax.plot(grid, mean, color=CURVE_COLOR, lw=2.0, zorder=3)

    ymax = max(float(high.max()) * 1.23, 1e-4)
    ax.text(-0.11, 1.02, panel_letter, transform=ax.transAxes, fontsize=9.5,
            fontweight="bold", va="bottom", ha="left")
    ax.set_title(rf"$\it{{Pdx1}} \rightarrow \it{{{target}}}$", loc="left", pad=5)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", color="#D7DCE2", lw=0.55, alpha=0.65)
    ax.tick_params(length=2.5, width=0.7, color="#626A73")

for ax in axes[::2]:
    ax.set_ylabel("Regulatory activity")
for ax in axes[-2:]:
    ax.set_xlabel("Pseudotime")

stage_handles = [Patch(facecolor=STAGE_FILL_COLORS[s], edgecolor=STAGE_COLORS[s], linewidth=0.8, label=s) for s in STAGE_ORDER]
curve_handle = Line2D([0], [0], color=CURVE_COLOR, lw=2, label="Smoothed regulatory activity")
fig.legend(handles=stage_handles + [curve_handle], loc="upper center", ncol=5,
           bbox_to_anchor=(0.5, 0.938), columnspacing=1.2, handlelength=1.7)

fig.subplots_adjust(left=0.105, right=0.985, bottom=0.075, top=0.875, hspace=0.42, wspace=0.30)

figure_base = OUTPUT_DIR / "pdx1_target_regulatory_strength_pseudotime_top50"
fig.savefig(figure_base.with_suffix(".svg"), bbox_inches="tight")
fig.savefig(figure_base.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(figure_base.with_suffix(".png"), dpi=600, bbox_inches="tight")
fig.savefig(figure_base.with_suffix(".tiff"), dpi=600, bbox_inches="tight",
            pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig)

print(f"Saved figure: {figure_base}.svg/.pdf/.png/.tiff")


### Figure interpretation and boundary

The figure is descriptive: it visualizes scDGRN-inferred Pdx1 edge strengths after a stringent per-cell global top-50 filter. The curves do not establish direct biochemical regulation or causal transitions. Because absent top-50 edges are encoded as zero, each curve jointly reflects edge strength and how often that edge enters the cell-specific top 50.


In [ ]:
expected_outputs = [
    selected_path,
    top50_path,
    trajectory_path,
    summary_path,
    figure_base.with_suffix(".svg"),
    figure_base.with_suffix(".pdf"),
    figure_base.with_suffix(".png"),
    figure_base.with_suffix(".tiff"),
]
missing = [str(path) for path in expected_outputs if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing expected outputs: {missing}")

print("Analysis completed successfully. Output files:")
for path in expected_outputs:
    print(f"- {path.name}: {path.stat().st_size / 1024:.1f} KiB")
